# Statistics for Data Analysts/Engineers — Module 4: Probability Distributions

In Module 3 we computed the probability of single events. A **probability distribution**
is a mathematical model describing *all possible* outcomes of a random phenomenon at
once -- and how likely each one is. We'll learn the four most important distributions and
how to use `scipy.stats` to compute probabilities from them, instead of only simulating.

## Table of contents
1. [Random variables and distributions: PMF, PDF, CDF](#sec1)
2. [Uniform distribution](#sec2)
3. [Binomial distribution](#sec3)
4. [Poisson distribution](#sec4)
5. [Normal distribution](#sec5)
6. [The 68-95-99.7 rule and standardization (z-score)](#sec6)
7. [Summary and exercises](#sec7)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Random variables and distributions: PMF, PDF, CDF

A **random variable** is a number whose value depends on the outcome of a random
experiment (e.g. "the number rolled on a die", "the spend of a random customer").
Distributions come in two flavors:

- **Discrete** — the variable takes a countable set of values (e.g. `0, 1, 2, 3...`).
  Described by a **PMF** (*Probability Mass Function*) -- a function returning
  `P(X = x)` for every possible value `x`.
- **Continuous** — the variable can take any value in a range of real numbers (e.g.
  height, time, spend). Described by a **PDF** (*Probability Density Function*) -- but
  BEWARE: `PDF(x)` is NOT `P(X = x)` (for a continuous distribution, the probability of
  hitting exactly one point is zero!). Probabilities for continuous variables are
  computed as the **area under the PDF curve** over some interval.

Both types have a **CDF** (*Cumulative Distribution Function*) -- `CDF(x) = P(X <= x)`,
the probability the random variable is less than or equal to `x`. `scipy.stats` gives
every distribution `.pmf()`/`.pdf()`, `.cdf()`, `.rvs()` (to draw samples), and `.ppf()`
(the inverse of the CDF -- "for what x is CDF(x) = p?").

<a id="sec2"></a>
## 2. Uniform distribution

A **uniform** distribution means every value in a given range is equally likely. Our
`age` column was generated exactly this way (`rng.integers(18, 70)`) -- every age from
18-69 had an equal chance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.figure(figsize=(8, 4))
plt.hist(customers["age"], bins=20, edgecolor="white", density=True)
plt.xlabel("Age")
plt.ylabel("Density")
plt.title("Customer age -- approximating a uniform distribution")
plt.show()


`scipy.stats.uniform` lets you compute exact theoretical probabilities (not just eyeball
a histogram). By default it describes a distribution on `[0, 1]`, so you pass `loc` (the
start of the range) and `scale` (the length of the range):

In [ ]:
age_distribution = stats.uniform(loc=18, scale=70 - 18)  # range [18, 70)

# P(age <= 40) theoretically
print(f"P(age <= 40) theoretically: {age_distribution.cdf(40):.3f}")
# the same thing empirically, from our data
print(f"P(age <= 40) empirically: {(customers['age'] <= 40).mean():.3f}")


<a id="sec3"></a>
## 3. Binomial distribution

The **binomial** distribution describes the number of "successes" in `n` independent
trials, each with the same success probability `p` (e.g. `n` coin flips, `n` promotional
emails sent, where each customer converts with probability `p`). This is a natural model
for our `converted` column.

In [ ]:
# If we send an offer to 20 customers from group A (P(conversion) = 0.10 by assumption),
# how many are likely to convert?
n_customers = 20
p_conversion = 0.10
conversion_distribution = stats.binom(n=n_customers, p=p_conversion)

# P(exactly 3 conversions)
print(f"P(exactly 3 conversions): {conversion_distribution.pmf(3):.3f}")
# P(at most 1 conversion)
print(f"P(at most 1 conversion): {conversion_distribution.cdf(1):.3f}")
# expected value = n * p
print(f"Expected number of conversions: {conversion_distribution.mean():.1f}")


In [ ]:
x = np.arange(0, n_customers + 1)
plt.figure(figsize=(8, 4))
plt.bar(x, conversion_distribution.pmf(x))
plt.xlabel("Number of conversions out of 20")
plt.ylabel("Probability (PMF)")
plt.title("Binomial distribution: n=20, p=0.10")
plt.show()


> 📊 **The binomial distribution underlies A/B testing**
>
> When we compute, in Module 14, how many observations you need for a reliable A/B test, the binomial distribution is always working in the background -- conversion/no conversion is a classic Bernoulli trial, and the sum of many such trials is a binomial distribution.

<a id="sec4"></a>
## 4. Poisson distribution

The **Poisson distribution** describes the number of events occurring in a fixed
interval of time (or space), when events happen at a known average rate `lambda`,
independently of each other. Typical examples: the number of orders per hour, the number
of server crashes per day, the number of customers entering a store per minute.

In [ ]:
# Suppose the shop gets 4 orders per hour on average (lambda=4)
orders_lambda = 4
orders_distribution = stats.poisson(mu=orders_lambda)

print(f"P(exactly 4 orders in an hour): {orders_distribution.pmf(4):.3f}")
print(f"P(0 orders in an hour): {orders_distribution.pmf(0):.3f}")
print(f"P(more than 8 orders in an hour): {1 - orders_distribution.cdf(8):.3f}")

x = np.arange(0, 15)
plt.figure(figsize=(8, 4))
plt.bar(x, orders_distribution.pmf(x))
plt.xlabel("Number of orders in an hour")
plt.ylabel("Probability (PMF)")
plt.title("Poisson distribution: lambda=4")
plt.show()


> ⚠️ **Poisson assumes a constant rate and independent events**
>
> The Poisson distribution assumes events happen at a constant, average rate and independently of each other (one order doesn't raise or lower the chance of the next). In a real shop that's a simplification -- e.g. a promotion might temporarily boost the order rate -- but as a starting point the Poisson model is very useful and surprisingly good for many "rare event" phenomena.

<a id="sec5"></a>
## 5. Normal distribution

The **normal** (Gaussian, "bell curve") distribution is the single most important
continuous distribution in statistics -- symmetric, fully described by two parameters:
the mean `mu` (the center, the peak of the curve) and the standard deviation `sigma` (how
spread out the data is around the mean). Our `spend_before` column was generated exactly
from a normal distribution (`N(220, 40)`).

In [ ]:
spend_before_distribution = stats.norm(loc=220, scale=40)

plt.figure(figsize=(8, 4))
plt.hist(customers["spend_before"], bins=30, density=True, edgecolor="white", alpha=0.6, label="Data (histogram)")

x = np.linspace(customers["spend_before"].min(), customers["spend_before"].max(), 200)
plt.plot(x, spend_before_distribution.pdf(x), color="red", linewidth=2, label="Theoretical PDF N(220, 40)")
plt.xlabel("Spend before the loyalty program")
plt.ylabel("Density")
plt.title("'Before' spend against the theoretical normal distribution")
plt.legend()
plt.show()


The histogram of our data lines up almost perfectly with the theoretical curve --
because that's literally how we generated it. In practice you never know the "true"
parameters -- you estimate them from a sample (`customers['spend_before'].mean()` and
`.std()`), which does exactly what Module 1 taught you.

<a id="sec6"></a>
## 6. The 68-95-99.7 rule and standardization (z-score)

For a normal distribution, a handy rule of thumb applies:
- about **68%** of observations fall within `mu ± 1*sigma`,
- about **95%** fall within `mu ± 2*sigma`,
- about **99.7%** fall within `mu ± 3*sigma`.

**Standardization** (z-score) converts any value `x` into the number of standard
deviations it is from the mean: `z = (x - mu) / sigma`. It lets you compare values from
different distributions on a common scale.

In [ ]:
mu = customers["spend_before"].mean()
sigma = customers["spend_before"].std(ddof=0)  # using population parameters (ddof=0) here
                                                 # since we're comparing to the theoretical definition

for k in [1, 2, 3]:
    share = ((customers["spend_before"] >= mu - k * sigma) & (customers["spend_before"] <= mu + k * sigma)).mean()
    print(f"Share within mu +/- {k} sigma: {share:.3f}")


In [ ]:
# the z-score of one specific customer
example_customer = customers.iloc[0]
z = (example_customer["spend_before"] - mu) / sigma
print(f"Customer's spend: {example_customer['spend_before']:.2f}")
print(f"Z-score: {z:.2f} standard deviations from the mean")

# scipy.stats.zscore does the same for a whole column at once
z_all = stats.zscore(customers["spend_before"])
print(z_all[:5])


> 📊 **The z-score is a preview of the tests in Modules 7-8**
>
> The test statistic in a z-test (and in practice often in a t-test too) is fundamentally a z-score -- "how many standard deviations does the observed result differ from what we'd expect under the null hypothesis". The exact same formula will come back in Module 7.

<a id="sec7"></a>
## 7. Summary and exercises

In this module we covered:
- the difference between a PMF (discrete distributions), a PDF (continuous
  distributions), and a CDF (both types),
- the uniform distribution (our `age` column),
- the binomial distribution -- the number of successes in `n` trials (conversions),
- the Poisson distribution -- the number of events in a fixed time window (orders per
  hour),
- the normal distribution -- the single most important continuous distribution
  (`spend_before`),
- the 68-95-99.7 rule and standardization (z-score).

> 📝 **Exercise 1: Binomial probability**
>
> Group B has `P(conversion) = 0.20`. If we send an offer to 15 group-B customers, what's the probability EXACTLY 5 of them convert? And what's the probability AT LEAST 5 convert?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
distribution_b = stats.binom(n=15, p=0.20)
print(f"Exactly 5: {distribution_b.pmf(5):.3f}")
print(f"At least 5: {1 - distribution_b.cdf(4):.3f}")
```

'At least 5' is the complement of 'at most 4' -- 1 - CDF(4), not 1 - CDF(5), since CDF(4) already includes every value <= 4.
</details>

> 📝 **Exercise 2: Poisson: probability of zero events**
>
> A server crashed on average 0.5 times a day (`lambda=0.5`). What's the probability of an entire week (7 days) with NO crashes at all? Hint: scale `lambda` up to a full week (`lambda * 7`) before computing `pmf(0)`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
weekly_lambda = 0.5 * 7
crash_distribution = stats.poisson(mu=weekly_lambda)
print(crash_distribution.pmf(0))
```
</details>

> 📝 **Exercise 3: Standardizing satisfaction**
>
> Compute the z-score of the `satisfaction` column (use `stats.zscore`). How many observations have `|z| > 2` (i.e. lie more than 2 standard deviations from the mean)? Compare this count to the 68-95-99.7 rule from section 6 -- is it close to the expected ~5%?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
z_satisfaction = stats.zscore(customers["satisfaction"])
outlier_share = (np.abs(z_satisfaction) > 2).mean()
print(f"Share with |z| > 2: {outlier_share:.3f}")
print("Expected under the 68-95-99.7 rule: about 1 - 0.95 = 0.05")
```
</details>

> 📝 **Exercise 4: A percentile from the normal distribution**
>
> Use `stats.norm(loc=mu, scale=sigma).ppf(0.90)` (where `mu`/`sigma` are the mean/std of `spend_before` from section 6) to compute the THEORETICAL 90th percentile of 'before' spend. Compare it with the EMPIRICAL percentile `customers['spend_before'].quantile(0.90)` from Module 1 -- are they close?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
distribution = stats.norm(loc=mu, scale=sigma)
theoretical_percentile = distribution.ppf(0.90)
empirical_percentile = customers["spend_before"].quantile(0.90)

print(f"Theoretical (from the normal distribution): {theoretical_percentile:.2f}")
print(f"Empirical (from the data): {empirical_percentile:.2f}")
```

These should come out very close, since spend_before genuinely comes from a normal distribution -- 'ppf' is exactly the inverse of 'cdf' from section 1.
</details>

> 🔥 **Challenge: simulation vs. theory**
>
> Draw (`stats.norm(loc=220, scale=40).rvs(size=10000, random_state=42)`) 10,000 values from the theoretical normal distribution. Plot, on one chart, a histogram of the drawn data AND a histogram of the real `spend_before` column (both with `density=True`, so they're comparable despite the different observation counts). Are the shapes similar, even though one dataset has 10,000 points and the other only 400?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
simulated = stats.norm(loc=220, scale=40).rvs(size=10_000, random_state=42)

plt.figure(figsize=(8, 4))
plt.hist(simulated, bins=40, density=True, alpha=0.5, label="Simulation (n=10000)")
plt.hist(customers["spend_before"], bins=30, density=True, alpha=0.5, label="Data (n=400)")
plt.xlabel("Spend before")
plt.ylabel("Density")
plt.legend()
plt.title("Simulation vs. real data")
plt.show()
```
</details>

## What's next?

In **Module 5** we'll meet one of the most important theorems in all of statistics: the
**Central Limit Theorem**. We'll see why the distribution of *sample means* approaches a
normal distribution even when the underlying data (like our right-skewed
`monthly_spend`) isn't normal at all -- and why this is the foundation of nearly all of
inferential statistics.